# tabular_agent — Phase 2: Feature Transforms & Encoding

The [sandbox notebook](phase2_sandbox_and_exploration.ipynb) built the isolation
boundary and the *read-only* tool `explore_dataframe`. This notebook adds the two
tools that actually **build features the model trains on**:

- **`transform_features`** (§5.1) — the *escape hatch*: the agent writes free
  `df → df` pandas/numpy code. Stateless, row-local math. Persisted and re-run at
  predict.
- **`fit_encoding`** (§5.2) — the *structured* tool for the one thing free code is
  deliberately forbidden to do: **learn a parameter from the target**
  (target-encoding), plus its non-leaky cousin (frequency-encoding).

The whole design turns on the line between those two. Low-jargon walkthrough; run
it cell by cell.

## 0. Setup — a small churn set

`age`, a right-skewed `income`, a `city` that really shifts churn, and the binary
`churn` target. Same flavour as Phase 1.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 2000

city = rng.choice(["berlin", "hamburg", "munich", "cologne", "leipzig"],
                  size=n, p=[0.30, 0.25, 0.20, 0.15, 0.10])
city_churn = {"berlin": 0.55, "hamburg": 0.25, "munich": 0.70,
              "cologne": 0.20, "leipzig": 0.45}
age = rng.integers(18, 80, size=n).astype(float)
income = rng.lognormal(mean=10.4, sigma=0.5, size=n)  # right-skewed

p = np.array([city_churn[c] for c in city])
p = np.clip(p + (age - 50) * 0.002 - (np.log(income) - 10.4) * 0.05, 0.02, 0.98)
churn = (rng.random(n) < p).astype(int)

df = pd.DataFrame({"city": city, "age": age, "income": income, "churn": churn})
X = df.drop(columns="churn")
y = df["churn"]
df.head()

## 1. Three ways the agent touches the data — one boundary

| Tool | Code? | Output | Persisted? | Can learn from `y`? |
|---|---|---|---|---|
| `explore_dataframe` | free | a *capped summary*, then discarded | no | — (read-only) |
| `transform_features` | free | a **DataFrame** → features | yes, as **code** | **no** (structurally blocked) |
| `fit_encoding` | *no* — typed call | encoded columns + fitted params | yes, as **params** | **yes**, but only **per-fold** |

The middle column is the whole point. Free code (`transform_features`) is allowed
to be arbitrary *because* it is structurally unable to learn state — the sandbox
blocks the model/sklearn libs (shown in the sandbox notebook). The one useful thing
that requires learning from the target — a category's target mean — is therefore
*not* reachable as free code, and is only offered through `fit_encoding`, where the
leakage-preventing per-fold discipline is baked in.

## 2. `transform_features` — stateless `df → df`

The agent writes a function `transform_features(df)` that returns a DataFrame. This
is exactly the row-local feature math from the report's examples: logs, ratios,
per-row combinations. No parameter is learned from the data — every output cell is a
function of that same row's inputs.

In [ ]:
from tabular_agent.transform import transform_features

feature_code = '''
import numpy as np
def transform_features(df):
    df = df.copy()
    df["log_income"] = np.log(df["income"])
    df["income_per_age"] = df["income"] / df["age"]
    return df
'''

out, summary = transform_features(X, feature_code)
print("new columns:", summary["new_columns"], "| rows:", summary["n_rows"])
out.head()

### The contract: preserve the row index

`transform_features` must return a DataFrame **with the same row index** — add or
replace columns, but don't reorder, drop, or add rows. That is what *makes* it
stateless and row-local (and keeps every row aligned with its label `y`). Violations
come back as a teaching error, not a silent misalignment.

In [ ]:
from tabular_agent.transform import SandboxError

bad = '''
def transform_features(df):
    return df[df["age"] > 40]   # drops rows -> breaks alignment with y
'''
try:
    transform_features(X, bad)
except SandboxError as e:
    print("rejected:", e)

### The structural guard

The same import whitelist as `explore`: no `lightgbm`/`xgboost`/`catboost`/`sklearn`.
So free feature code **cannot train a model or learn state** — it can only do
row-local math. (That is the property that lets us re-run it blindly at predict.)

In [ ]:
cannot_learn = '''
import sklearn
def transform_features(df):
    return df
'''
try:
    transform_features(X, cannot_learn)
except SandboxError as e:
    print("rejected:", e)

## 3. Persisted as *code*, replayed at predict

A `transform_features` step is stored in the pipeline as **the code string** — not
the columns it produced. At predict time the pipeline simply re-runs it in the
sandbox. Because it is stateless, replaying is trivially reproducible: no stored
statistics, no train/serve skew possible.

In [ ]:
from tabular_agent.pipeline import Pipeline

pipe = Pipeline("binary")
pipe.add_transform(feature_code)

# Survives serialization with only the code inside it:
manifest = pipe.to_dict()
print("stored step keys:", list(manifest["steps"][0].keys()))

restored = Pipeline.from_dict(manifest)
replayed = restored.transform(X)
print("replayed new columns:", [c for c in replayed.columns if c not in X.columns])

## 4. `fit_encoding` — the structured, stateful tool

Now the other side of the boundary: turning a **category** into a number the model
can split on. Two methods, and the difference between them is the whole leakage story.

### 4a. Target encoding — learns from `y`, so it must be per-fold

`city → mean(churn | city)`. This is target-informed, so used naively it leaks the
label into the feature. `fit_encoding(method="target")` returns **two** things:

- `train_feats` — **out-of-fold** encoded columns, for *honest training*;
- `final_enc` — an encoder fit on **all** rows, for the pipeline / predict.

(See the [target-encoding tutorial](target_encoding_step_by_step.ipynb) for why OOF
is the honest choice.)

In [ ]:
from tabular_agent.encoders import fit_encoding
from tabular_agent.cv import make_folds

folds = make_folds(y, "binary", n_splits=5, seed=42)
train_feats, final_enc = fit_encoding(X, ["city"], "target", y=y, folds=folds)

print("train (OOF) columns:", list(train_feats.columns))
# The final encoder's learned mapping (fit on all rows):
print("learned city -> churn-mean:")
{k: round(v, 3) for k, v in final_enc.mapping_["city"].items()}

### 4b. Frequency encoding — uses only `X`, so no leakage to guard

`city → how common it is`. It never looks at the target, so there is nothing to
leak and no per-fold machinery is needed. Its job is to make string columns usable
by models that don't consume raw categoricals. Unknown categories at predict time
map to `0` (unseen ⇒ frequency zero).

In [ ]:
freq_feats, freq_enc = fit_encoding(X, ["city"], "frequency")
print("no y, no folds needed. columns:", list(freq_feats.columns))

# Unknown category at predict time -> 0.0
unseen = pd.DataFrame({"city": ["berlin", "atlantis"]})
freq_enc.transform(unseen)[["city", "city_freq"]]

## 5. Putting it together — transform + encode → train → predict

A minimal end-to-end run showing both new tools inside the same pipeline. We train
on the **OOF** target-encoded column plus the stateless features; the *bundle* stores
the transform **code** and the **final** encoder, and `predict` replays both on raw
data that has no target column.

In [ ]:
from tabular_agent.models import train_cv
from tabular_agent.pipeline import save_bundle, load_bundle, predict_bundle
from tabular_agent.runctx import RunContext

# Stateless features (OOF-independent) computed for training:
X_feat, _ = transform_features(X, feature_code)

# Assemble the training matrix: stateless features + OOF target-encoding.
train_X = pd.concat(
    [X_feat[["log_income", "income_per_age", "age"]], train_feats], axis=1
)
feature_names = list(train_X.columns)

res = train_cv(train_X, y, folds, "binary", "auc", model_type="lgbm")
cv_auc = float(np.mean(res.fold_metrics))
print("mean fold AUC:", round(cv_auc, 4), "| features:", feature_names)

In [ ]:
# Build the reproducible pipeline: transform code THEN the fitted encoder.
pipe = Pipeline("binary")
pipe.add_transform(feature_code)   # replayed at predict
pipe.add_encoder(final_enc)        # applied at predict (fit on all rows)

ctx = RunContext("/tmp/tabular_phase2_demo_run")
bundle_dir = save_bundle(ctx, pipe, [res.final_model], np.array([1.0]),
                         feature_names, "binary")

bundle = load_bundle(bundle_dir)
# Raw new customers -- categories only, NO target column:
new_raw = pd.DataFrame({
    "city":   ["munich", "cologne", "berlin"],
    "age":    [30.0, 65.0, 45.0],
    "income": [45000.0, 90000.0, 60000.0],
})
preds = predict_bundle(bundle, new_raw)
print("predicted churn probability:", np.round(preds, 3))
# munich (high-churn city) should rank above cologne (low-churn city):
print("munich > cologne:", preds[0] > preds[1])

## 6. Recap

- **`transform_features`** — free `df → df` code for stateless, row-local features.
  Persisted as *code*, replayed at predict; structurally can't learn state. The row-
  index contract is what keeps it stateless.
- **`fit_encoding`** — the *only* path to a target-informed feature, and it enforces
  the leakage-safe discipline: **out-of-fold** training columns + a **final** encoder
  fit on all rows for predict. `frequency` mode is the non-leaky, `X`-only encoder.
- **The pipeline** carries both — transform *code* and encoder *params* — so `predict`
  reproduces the exact feature matrix from raw data, no target needed, no skew.

**Next (Phase 2 tail):** wrap these primitives (`inspect_data`, `transform_features`,
`fit_encoding`, `train_tabular_model`, `blend_models`, `predict`, `finalize_run`) as
the actual **MCP server** — the stateless, file-based tool surface from §3. The
optional §18 SHAP explainability comes after.